# Field-Weighted Citation Impact (FWCI)
FWCI is the primary impact metric in the manuscript: it normalizes citation performance against publications from the same field and publication year, so it needs no separate adjustment for article age or field.

All comparisons are run at two levels: (1) a one-sided comparison of sharing vs. non-sharing articles, and (2) a two-sided comparison across the three sharing granularities (FIXATION, TRIAL, PARTICIPANT).

In [ ]:
from typing import Literal, Dict
from itertools import combinations
from copy import deepcopy

import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from helpers import dataset
from helpers.config import (
    CLASS_COLORS, SHARING_CLASS_ORDER, BINARY_FEATURES,
    TITLE_FONT, AXIS_TITLE_FONT, AXIS_TICK_FONT, LEGEND_FONT, FONT_FAMILY,
    VENUE_IMPACT_METRIC, VENUE_IMAPCT_METRIC_NAME,
)
from helpers.plotting import save_figure

pio.renderers.default = "browser"

combined, FEATURES_DF, CITATIONS_DF = dataset.load_or_build()

### (1) FWCI
#### (1A) Sharing / Non-Sharing Articles

In [ ]:
not_sharing_fwci = combined.loc[~combined["is_sharing_data"], "FieldWeightedCitationIndex"]
sharing_fwci = combined.loc[combined["is_sharing_data"], "FieldWeightedCitationIndex"]

# check normality of FWCI distributions:
print("Is FWCI normally distributed for non-sharing articles?")
fwci_non_sharing_shapiro = stats.shapiro(not_sharing_fwci)
print(f"{"YES" if fwci_non_sharing_shapiro.pvalue > 0.05 else "NO"} (W={fwci_non_sharing_shapiro.statistic:.3f}, p={fwci_non_sharing_shapiro.pvalue:.4f})\n")

print("Is FWCI normally distributed for sharing articles?")
fwci_sharing_shapiro = stats.shapiro(sharing_fwci)
print(f"{"YES" if fwci_sharing_shapiro.pvalue > 0.05 else "NO"} (W={fwci_sharing_shapiro.statistic:.3f}, p={fwci_sharing_shapiro.pvalue:.4f})")

In [ ]:
fwci_pair_test = stats.mannwhitneyu(not_sharing_fwci, sharing_fwci, alternative="less")

# calculate effect size (rank-biserial correlation / CLES)
n1, n2 = len(not_sharing_fwci), len(sharing_fwci)
U_nonshare = fwci_pair_test.statistic
U_share = n1 * n2 - U_nonshare
rank_biserial_corr = (U_share - U_nonshare) / (n1 * n2)
cles = U_share / (n1 * n2)

print("Share / Non-Share FWCI Comparison:")
print("Mann-Whitney U Test:" + f"\tU={fwci_pair_test.statistic},\tp={fwci_pair_test.pvalue:.4f}")
print("{0: <20}".format("Effect Sizes:") + f"\tRBC: {rank_biserial_corr:.3f}\tCLES: {cles:.3f}")

# re-run statistical test with `pingouin` for robustness:
pg.mwu(not_sharing_fwci, sharing_fwci, alternative="less")

#### (1B) Data Sharing Class

In [ ]:
# check normality of FWCI distributions for each sharing class:
share_group_fwci = []
for share_group in ["FIXATION", "TRIAL", "PARTICIPANT"]:
    group_fwci = (
        combined
        .loc[combined["data_sharing_class"] == share_group, "FieldWeightedCitationIndex"]
        .dropna()
        .reset_index(drop=True)
        .rename(share_group)
    )
    share_group_fwci.append(group_fwci)
    shapiro_result = stats.shapiro(group_fwci)
    print(f"Is FWCI normally distributed for {share_group} sharing articles?")
    print(f"{"YES" if shapiro_result.pvalue > 0.05 else "NO"} (W={shapiro_result.statistic:.3f}, p={shapiro_result.pvalue:.4f})\n")

# run Kruskal-Wallis H-test:
fwci_sharegroup_test = stats.kruskal(*share_group_fwci)
fwci_sharegroup_test

#### Visualizing Share/Non-Share FWCI Comparison

In [ ]:
suplot_titles = ["Sharing vs. Non-Sharing", "Different Types of Data Sharing"]
fwci_fig = make_subplots(
    rows=1, cols=len(suplot_titles), column_titles=suplot_titles,
    shared_yaxes=True, horizontal_spacing=0.025, vertical_spacing=0.025,
)

# (1) Sharing vs. Non-Sharing
left_subplot_data_long = (
    combined
    .melt(
        id_vars=['is_sharing_data'],
        value_vars=['FieldWeightedCitationIndex'],
        var_name='name',
        value_name='value'
    )
    .replace({
        'FieldWeightedCitationIndex': 'FWCI',
    })
)
max_y = left_subplot_data_long["value"].max()
for is_share in left_subplot_data_long["is_sharing_data"].unique():
    subset = left_subplot_data_long[left_subplot_data_long["is_sharing_data"] == is_share]
    subset_name = 'Sharing' if is_share else 'Not Sharing'
    color = CLASS_COLORS[subset_name.upper()]
    fwci_fig.add_trace(
        row=1, col=1,
        trace=go.Violin(
            x=subset['name'], y=subset['value'],
            name=subset_name, legendgroup=subset_name, scalegroup=subset_name,
            side='positive' if is_share else 'negative',
            fillcolor=color, line_color=color,
            width=0.9, spanmode="hard",
            points=False, pointpos=0, jitter=0.5,
            box=dict(visible=False, width=0.5, line=dict(color="black")),
            meanline=dict(visible=False, color='black'),
        )
    )
    # annotate half-violin
    fwci_fig.add_annotation(
        row=1, col=1,
        text=subset_name, font={**AXIS_TITLE_FONT, "color": color},
        x=0.25 if is_share else -0.2, xanchor="center", xref="x",
        y=0.925 * max_y, yanchor="middle", yref="paper",
        showarrow=False,
    )

# add significance asterisk
fwci_fig.add_annotation(
    row=1, col=1,
    x="FWCI", xanchor="center", xref="x",
    y=max_y * 1.05, yanchor="middle", yref="paper",
    text="*", font=TITLE_FONT, font_size=30,
    showarrow=False,
)
# add separation line
fwci_fig.add_shape(
    row=1, col=1,
    type="line", line=dict(color="black", width=2, dash="dash"),
    x0="FWCI", x1="FWCI", y0=0, y1=max_y,
)

# (2) Data Sharing Class
right_subplot_data_long = (
    combined
    .loc[combined["data_sharing_class"].isin(["FIXATION", "TRIAL", "PARTICIPANT"])]
    .melt(
        id_vars=['data_sharing_class'],
        value_vars=['FieldWeightedCitationIndex'],
        var_name='name',
        value_name='value'
    )
    .replace({
        'FieldWeightedCitationIndex': 'FWCI',
    })
)
for i, share_class in enumerate(right_subplot_data_long["data_sharing_class"].unique()):
    subset = right_subplot_data_long[right_subplot_data_long["data_sharing_class"] == share_class]
    color = CLASS_COLORS[share_class]
    fwci_fig.add_trace(
        row=1, col=2,
        trace=go.Violin(
            x=subset['data_sharing_class'], y=subset['value'],
            name=share_class, legendgroup=share_class, scalegroup=share_class,
            side='positive', points='all', pointpos=-0.25, jitter=0.2,
            fillcolor=color, line=dict(color=color, width=1.5),
            width=1.0, spanmode="hard",
            box=dict(visible=False, width=0.5, line=dict(color="gray")),
            meanline=dict(visible=False, color='gray'),
        ))
    # annotate half-violin
    fwci_fig.add_annotation(
        row=1, col=2,
        text=share_class.title(),
        font={**AXIS_TITLE_FONT, "color": color},
        x=i, xanchor="center", xref="x",
        y=0.925 * max_y, yanchor="middle", yref="paper",
        showarrow=False,
    )

# add n.s. mark
fwci_fig.add_annotation(
    row=1, col=2,
    x=1.0, xanchor="center", xref="x",
    y=max_y * 1.05, yanchor="middle", yref="paper",
    text="n.s.", font=TITLE_FONT,
    showarrow=False,
)


# update layout and show
for ann in fwci_fig.layout.annotations:
    if ann.text not in suplot_titles:
        continue
    ann_xref = 'x' if ann.text == suplot_titles[0] else 'x2'
    ann.update(dict(
        font=TITLE_FONT,
        x=-0.5, xanchor="left", xref=ann_xref,
        y=1.0, yanchor="top", yref="paper",
))
fwci_fig.update_xaxes(
    title=None, showticklabels=False, tickfont=AXIS_TICK_FONT, zeroline=False,
)
fwci_fig.update_yaxes(
    showgrid=True, gridcolor='lightgrey', gridwidth=1.5,
    zeroline=False,
)
fwci_fig.update_yaxes(
    row=1, col=1,
    range=[-0.5, max_y * 1.2],
    title=dict(text="<b>FWCI</b>", font=AXIS_TITLE_FONT, standoff=5),
    tickfont=AXIS_TICK_FONT,
)
fwci_fig.update_layout(
    width=1000, height=500,
    title=dict(
        text="<b>FWCI Distribution for Data Sharing Class</b>",
        font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top"
    ),
    violinmode='overlay',
    violingap=0,
    legend=dict(visible=False,),
    margin=dict(t=50, b=5, l=50, r=10, pad=0),
    template="plotly_white",
)
fwci_fig.show()

In [ ]:
# flip to True to export this figure into `output/`
if False:
    save_figure(fwci_fig, "fwci_data_sharing.png", width=1000, height=500)

### Adjusting for Other Citation Predictors
Publication age and venue impact are deliberately excluded from this model: FWCI already normalizes for field and publication year by construction.

In [ ]:
fwci_model_df = pd.concat(
    [FEATURES_DF, np.log(combined["FieldWeightedCitationIndex"] +1)], axis=1
).rename(columns={
    "Is Sharing Data": "shares_data",
    "Has Preprint": "has_preprint",
    "Is Open Access": "is_oa",
    "Has US Author": "has_us",
    "log(Weeks Since Pub.)": "log_weeks",
    "log(Number of Authors)": "log_authors",
    "Venue Impact": "venue_impact",
    "FieldWeightedCitationIndex": "log_fwci",
})

covariate_model = smf.ols(
    "log_fwci ~ shares_data + is_oa + has_preprint + has_us + log_authors",
    # we exclude log_weeks + log_venue_impact - those are baked into FWCI by design
    data=fwci_model_df,
).fit()

beta = covariate_model.params["shares_data"]
ci_low, ci_high = covariate_model.conf_int().loc["shares_data"]
print(
    f"\nData-sharing effect adjusted for publication age, venue impact, author count, US authorship, open access, and pre-print publication:"
    f"\n\tbeta = {beta:.3f}  (95% CI [{ci_low:.3f}, {ci_high:.3f}]),  p = {covariate_model.pvalues['shares_data']:.3f}"
    f"\n\tmultiplicative citation advantage = exp(beta) = {np.exp(beta):.3f}× ({(np.exp(beta) - 1) * 100:+.1f}%)"
)
covariate_model.summary()